# 01 · A happy-path run through the agents

One contract document, end to end, through the **real** pipeline graph —
watching every agent's station: what it read, what it wrote, and why the
router went where it went next.

**What you'll see:** the clean six-node path
`intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-document`,
the per-node narration table, the confidence bands that did (and did not)
fire, and the artifacts the run leaves on disk (manifest, catalog row,
hash-chained audit entries, archive bin).

**Honesty label:** the graph, routing thresholds, bins, catalog, and audit
chain are the REAL code paths. The LLMs are deterministic mocks — the same
`FakeLangChainLLM` + scripted-client seam the test suite uses
(`src/tests/conftest.py`). Outputs below are what the pipeline *actually
produced* under those mocks; nothing is hand-written. With a real
`OPENROUTER_API_KEY` the same run would hit live models (nondeterministic
confidences, real cost, Langfuse traces).

Companion module: `notebooks/pipeline_lab.py` (the bench). Plan of record:
`notebooks/PLAN.md`.

## Setup — the lab bench

In [1]:
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab

## The document

A plain-text contract. (The real intake also handles PDF/DOCX/images via the
transcriber and image agents — see notebook 00.)

In [2]:
DOC = """MASTER SERVICES AGREEMENT

This Master Services Agreement ("Agreement") is entered into as of
2024-01-01 by and between Acme Corp ("Provider") and Beta LLC ("Client").

1. SERVICES. Provider shall provide legal-document processing services.
2. TERM. This Agreement continues until terminated by either party with
   thirty (30) days written notice.
3. GOVERNING LAW. This Agreement is governed by the laws of Delaware.
"""
print(DOC)

MASTER SERVICES AGREEMENT

This Master Services Agreement ("Agreement") is entered into as of
2024-01-01 by and between Acme Corp ("Provider") and Beta LLC ("Client").

1. SERVICES. Provider shall provide legal-document processing services.
2. TERM. This Agreement continues until terminated by either party with
   thirty (30) days written notice.
3. GOVERNING LAW. This Agreement is governed by the laws of Delaware.



## The run

`lab_sandbox()` redirects the pipeline's filesystem to a temp dir and
installs the mock seam; `run_document` drives the real `run_pipeline` while
recording each node's state delta. The canned classification says
`contract` at **0.98** confidence — above the high band — so the clean path
should stay clean.

In [3]:
env = lab.open_sandbox()
lab.script_client(env["client"], judge=lab.JUDGE_COMPLETE)  # lane-B fuel, unused on this path
result = lab.run_document(
    env,
    DOC,
    matter_id="LAB-MATTER-001",
    filename="msa_acme_beta.txt",
    classification=lab.CLASSIFY_CONTRACT_HIGH,   # sorter says: contract, 0.98
    extraction=lab.EXTRACT_HIGH,                 # specialist returns typed fields, 0.96
)
final = result["final"]
print("final stage:", final["stage"])
print("doc_id:      ", final["doc_id"])
lab.show_path(result["steps"])

final stage: archived
doc_id:       418bb1e2-8c1d-427f-9767-4b800b028619
intake-document → classify-document → extract-fields → compile-report → write-catalog → archive-document


**`archived`.** Six stations, no detours. Now the station-by-station story.

## Station-by-station: what each agent did

Each entry shows the node, the agent (or agents) that power it, its role,
and the exact state fields it changed (the step log is captured from the
real graph execution, not reconstructed).

In [4]:
lab.show_steps(result["steps"])

 1. intake-document  [pdf_transcriber / image_extractor (conditional)]
    role: Reads the raw file, extracts text + page images, assigns doc_id
    file_sha256: None → '37b553862639531d118f4cb66bd3c467636134ecc166ac75ca3c2d6209eb9aa1'
    doc_id: '' → '418bb1e2-8c1d-427f-9767-4b800b028619'
    doc_text: '<0 chars>' → '<419 chars>'
    size_bytes: None → 419
    stage: 'inbox' → 'processing'

 2. classify-document  [sorter (+ sorter_reviewer Lane A, judge-classification)]
    role: Labels doc_type + confidence; bands decide what happens next
    classification_confidence: None → 0.98
    classification_attempts: 0 → 1
    contract_subtype: None → 'other'
    stage: 'processing' → 'classified'
    classification_guardrail: None → []
    doc_type: None → 'contract'

 3. extract-fields  [class specialist (+ judge/arbiter Lane B)]
    role: Fills the class schema's typed fields
    conflict_details: None → []
    extraction_confidence: None → 0.96
    extraction_attempts: 0 → 1
    extract

## Why no judge? The band math

The judge/arbiter lane (KANBAN-063) fires only when extraction confidence
lands in the ambiguous band `[low, judge_band_high)`. Our specialist
returned **0.96** — above the band — so `judge_gate` said no. The bands come
live from `pipeline.config` (the same read `graph.routing` does):

In [5]:
bands = lab.band_report()
ext_conf = final["extraction_confidence"]
cls_conf = final["classification_confidence"]
print(f"thresholds (live):      {bands}")
print(f"classification {cls_conf} >= high {bands['high']}      -> clean classify, no retry, no Lane A")
in_band = bands["low"] <= ext_conf < bands["judge_band_high"]
print(f"extraction    {ext_conf} vs band [{bands['low']}, {bands['judge_band_high']}) -> judge_gate fired: {in_band}")

thresholds (live):      {'high': 0.95, 'low': 0.7, 'judge_band_high': 0.85}
classification 0.98 >= high 0.95      -> clean classify, no retry, no Lane A
extraction    0.96 vs band [0.7, 0.85) -> judge_gate fired: False


## What the run left behind

Everything The-Mailroom visualizer (and the audit tooling) reads:
manifest JSON, the SQLite catalog (`matters` / `documents` / `audit_log`),
and the archive bin layout `matter_id/doc_type/file`.

In [6]:
lab.show_artifacts(env["base_dir"])

manifests: {
  "418bb1e2-8c1d-427f-9767-4b800b028619.json": {
    "doc_id": "418bb1e2-8c1d-427f-9767-4b800b028619",
    "matter_id": "LAB-MATTER-001",
    "original_filename": "msa_acme_beta.txt",
    "stage": "archived",
    "doc_type": "contract",
    "contract_subtype": "other",
    "classification_confidence": 0.98,
    "classification_attempts": 1,
    "extracted_data": {
      "parties": [
        "Acme Corp",
        "Beta LLC"
      ],
      "effective_date": "2024-01-01",
      "reasoning": null,
      "document_name": null,
      "term_length": null,
      "termination_clauses": [],
      "governing_law": null,
      "key_obligations": [],
      "contract_value": null,
      "renewal_terms": null,
      "_report": {
        "summary": "Matter record compiled by the mock reporter (lab).",
        "doc_type": "contract",
        "contract_subtype": "other",
        "extracted_data": {
          "parties": [
            "Acme Corp",
            "Beta LLC"
          ],
          

## The final state, field by field

In [7]:
for key in (
    "doc_id", "matter_id", "stage", "doc_type", "contract_subtype",
    "classification_confidence", "extracted_data", "extraction_confidence",
    "trace_id", "review_decision", "escalation_reason",
):
    print(f"{key:28s} {final.get(key)!r}")

doc_id                       '418bb1e2-8c1d-427f-9767-4b800b028619'
matter_id                    'LAB-MATTER-001'
stage                        'archived'
doc_type                     'contract'
contract_subtype             'other'
classification_confidence    0.98
extracted_data               {'parties': ['Acme Corp', 'Beta LLC'], 'effective_date': '2024-01-01', 'reasoning': None, 'document_name': None, 'term_length': None, 'termination_clauses': [], 'governing_law': None, 'key_obligations': [], 'contract_value': None, 'renewal_terms': None, '_report': {'summary': 'Matter record compiled by the mock reporter (lab).', 'doc_type': 'contract', 'contract_subtype': 'other', 'extracted_data': {'parties': ['Acme Corp', 'Beta LLC'], 'effective_date': '2024-01-01', 'document_name': None, 'term_length': None, 'termination_clauses': [], 'governing_law': None, 'key_obligations': [], 'contract_value': None, 'renewal_terms': None}, 'classification_confidence': 0.98, 'extraction_confidence': 0.96}}
e

## Cleanup

Close the sandbox — the temp dir (bins, catalog, archive) is removed and every patched seam is restored.

In [8]:
lab.close_sandbox(env)

## Where to go next

- **00 pipeline_anatomy** — the full map: 13 nodes, all 15 agents, 7 doc classes
- **02 routing_dynamics** — re-run this exact document at confidences 0.98 → 0.10 and watch the bands steer different paths
- **03 review_lanes** — what happens when the judge DOES fire (and fails), and when the sorter_reviewer overrides
- **06 outputs_and_audit** — a deep tour of the manifest/catalog/audit artifacts this run just wrote